In [0]:
# Cell 1: setup
from pyspark.sql import functions as F
from pyspark.sql.functions import col

CATALOG = "city_lens"
SILVER = f"{CATALOG}.silver"
TARGET = f"{CATALOG}.gold.happenings_enriched"
RAIN_THRESHOLD = 60      # percent
CLOSURE_RADIUS_M = 500

spark.conf.set("spark.sql.session.timeZone", "Europe/Amsterdam")

def distance_km(lat1, lon1, lat2, lon2):
    dlat = F.radians(lat2 - lat1)
    dlon = F.radians(lon2 - lon1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371 * F.asin(F.sqrt(a))

In [0]:
# Cell 2: focus day and status per happening
today   = F.current_date()
start_d = F.to_date("start_local")
end_d   = F.to_date("end_local")
scheduled = col("time_basis") == "scheduled"

h = (
    spark.table(f"{SILVER}.city_happenings")
    .withColumn("focus_date",
        F.when(scheduled,
               F.when(end_d < today, start_d).otherwise(F.greatest(start_d, today)))
         .otherwise(col("published_date")))
    .withColumn("status",
        F.when(scheduled & (end_d < today), "ended")
         .when(scheduled & (start_d > today), "upcoming")
         .when(scheduled, "active")
         .when(col("published_date") >= F.date_sub(today, 14), "recent")
         .otherwise("older"))
)
display(h.groupBy("source", "status").count().orderBy("source", "status"))

In [0]:
# Cell 3: rain risk. Daytime max rain chance per day
daily_weather = (
    spark.table(f"{SILVER}.weather_hourly")
    .where(F.hour("hour_local").between(7, 22))
    .groupBy("city_key", col("date_local").alias("focus_date"))
    .agg(F.max("rain_chance_pct").alias("day_rain_pct"),
         F.round(F.max("temperature_c"), 1).alias("day_max_c"))
)

h = h.join(daily_weather, ["city_key", "focus_date"], "left")

In [0]:
# Cell 4: closure nearby. Active NDW road closures within 500 m on the focus day
closures = (
    h.where((col("source") == "ndw_melvin") & (col("closure_level") == "road_closed") & col("location_ok"))
        .select(col("city_key").alias("c_city"),
             col("happening_id").alias("c_id"),
             col("lat").alias("c_lat"), col("lon").alias("c_lon"),
             F.to_date("start_local").alias("c_start"),
             F.to_date("end_local").alias("c_end"))
)

near = (
    h.where(col("location_ok"))
     .select("happening_id", "city_key", "lat", "lon", "focus_date")
     .join(closures,
           (F.col("city_key") == col("c_city")) &
           (F.abs(col("lat") - col("c_lat")) < 0.006) &      # quick pre-filter (~650 m)
           (F.abs(col("lon") - col("c_lon")) < 0.01) &
           (col("happening_id") != col("c_id")) &
           col("focus_date").between(col("c_start"), col("c_end")))
     .withColumn("dist_m", distance_km(col("lat"), col("lon"), col("c_lat"), col("c_lon")) * 1000)
     .where(col("dist_m") <= CLOSURE_RADIUS_M)
     .groupBy("happening_id")
     .agg(F.count("*").alias("closures_nearby"),
          F.round(F.min("dist_m")).cast("int").alias("nearest_closure_m"))
)

h = h.join(near, "happening_id", "left").fillna({"closures_nearby": 0})

In [0]:
# Cell 5: badges, write gold, check
gold = (
    h
    .withColumn("badge_rain_risk",      scheduled & (col("day_rain_pct") >= RAIN_THRESHOLD))
    .withColumn("badge_closure_nearby", col("closures_nearby") > 0)
    .withColumn("badge_permit_only",    col("category") == "event_permit")
    .withColumn("badge_rain_risk",      F.coalesce("badge_rain_risk", F.lit(False)))
    .withColumn("refreshed_at",         F.current_timestamp())
)

(gold.write.format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(TARGET))

display(spark.sql(f"""
    SELECT category, status,
           count(*)                                         AS happenings,
           sum(CASE WHEN badge_rain_risk      THEN 1 ELSE 0 END) AS rain_risk,
           sum(CASE WHEN badge_closure_nearby THEN 1 ELSE 0 END) AS closure_nearby,
           sum(CASE WHEN badge_permit_only    THEN 1 ELSE 0 END) AS permit_only
    FROM {TARGET}
    GROUP BY category, status
    ORDER BY category, status
"""))